# Tehran Real Estate Valuation
**Author:** Haadi Jafari | **Dataset:** Tehran Housing Market (Kaggle)

---

## Executive Overview & Economic Framework
This project presents an end-to-end regression system to estimate residential real estate prices in **Tehran, Iran** from structural and geographic property attributes (Area, Bedrooms, Parking, Warehouse, Elevator, Address).

### Inflation Hedging via Intrinsic USD Valuation:
In high-inflation economies, property prices denominated in local currency (Iranian Rial / Toman) experience continuous nominal appreciation, causing nominal pricing models to drift and decay rapidly.
- **Intrinsic USD Baseline**: By training our predictive models directly on **`Price(USD)`**, we model the intrinsic economic purchasing power of real estate assets.
- **Dynamic Real-Time Conversion**: The model estimates the property value in USD; this intrinsic valuation is then coupled with an environment-configurable exchange rate (`USD_TO_TOMAN_RATE`) to project real-time Toman market prices for any macroeconomic timeframe.

### Key Engineering Pillars:
1. **Zero Data Leakage**: All imputations, scaling, and categorical encodings are strictly encapsulated inside scikit-learn `Pipeline` and `ColumnTransformer` structures.
2. **Target Normalization**: Logarithmic transformation ($\log(1 + \text{Price}_{USD})$) normalizes target skewness from **4.77** to **0.06**, guaranteeing Gaussian residuals and homoscedasticity.
3. **Rigorous Multi-Model Benchmark**: Systematic evaluation of diverse algorithms under 5-Fold Cross-Validation and unseen holdout test splits.
4. **Comprehensive Visual Diagnostics**: Complete suite of exploratory data analysis (EDA) and 4-in-1 residual diagnostic dashboards.

---
## 1. Setup & Environment Initialization
We import standard scientific and machine learning libraries, set visual aesthetics, and load environment variables.

In [ ]:
import os
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
from dotenv import load_dotenv
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, LinearRegression, RidgeCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load environment configuration (e.g. USD_TO_TOMAN_RATE)
load_dotenv()
DEFAULT_USD_TO_TOMAN: float = float(os.getenv("USD_TO_TOMAN_RATE", "220000"))

# Configure publication-quality visual aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

print(
    f"Environment initialized! Configured USD/Toman exchange rate: {DEFAULT_USD_TO_TOMAN:,.0f} Tomans/USD"
)

---
## 2. Data Ingestion & Domain-Grounded Cleaning

### Domain Validation Rules:
1. **Formatting**: Strip comma thousands-separators from `Area` and coerce to numeric.
2. **Physical Boundaries**: Residential properties in Tehran typically range between $15\text{ m}^2$ and $1,000\text{ m}^2$. Listings where area was corrupted due to data entry copy-paste errors (e.g., total price typed into area field $> 1,000,000,000$) are filtered out.
3. **Address Validity**: Listings missing neighborhood addresses ($23$ instances) are removed.
4. **Amenities**: Binary indicators (`Parking`, `Warehouse`, `Elevator`) are cast to standard integers ($0/1$).

In [ ]:
# Load raw dataset
data_path = Path("data/housePrice.csv")
df_raw = pd.read_csv(data_path)
print(f"Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
def clean_housing_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean raw housing data according to real estate domain rules."""
    cleaned = df.copy()

    # 1. Format Area: remove commas and cast to numeric
    if cleaned["Area"].dtype == object:
        cleaned["Area"] = cleaned["Area"].astype(str).str.replace(",", "", regex=False)
    cleaned["Area"] = pd.to_numeric(cleaned["Area"], errors="coerce")

    # 2. Filter missing addresses, areas, or prices
    cleaned = cleaned.dropna(subset=["Address", "Area", "Price(USD)"])
    cleaned["Address"] = cleaned["Address"].astype(str).str.strip()

    # 3. Filter physically impossible residential areas (15 to 1,000 m²)
    cleaned = cleaned[(cleaned["Area"] >= 15.0) & (cleaned["Area"] <= 1000.0)]

    # 4. Filter non-positive prices
    cleaned = cleaned[cleaned["Price(USD)"] > 0]

    # 5. Convert boolean amenities to integers
    for col in ["Parking", "Warehouse", "Elevator"]:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].astype(int)

    return cleaned.reset_index(drop=True)


df_clean = clean_housing_data(df_raw)
print(
    f"Cleaned dataset: {df_clean.shape[0]} records retained ({len(df_clean) / len(df_raw) * 100:.2f}%)"
)
df_clean.describe()

---
## 3. Exploratory Data Analysis (EDA)

We explore the distribution of real estate prices in USD, property dimensions, neighborhood tiers, and amenity valuations.

### 3.1 Target Distribution: Raw USD vs. Log-Transformed USD
Raw property prices in USD exhibit extreme right-skewness ($4.77$). Logarithmic transformation $\log(1 + \text{Price}_{USD})$ yields a symmetric, near-Gaussian bell curve with skewness of $0.06$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

raw_usd = df_clean["Price(USD)"]
log_usd = np.log1p(raw_usd)

# Raw USD Distribution
sns.histplot(raw_usd / 1e3, kde=True, ax=axes[0], color="#2b5c8f", bins=40)
axes[0].set_title(
    f"Raw Price(USD) [k USD] (Skewness: {raw_usd.skew():.2f})", fontsize=12, fontweight="bold"
)
axes[0].set_xlabel("Price in Thousands of USD ($k)")
axes[0].set_ylabel("Frequency")
axes[0].axvline(
    (raw_usd / 1e3).median(),
    color="red",
    linestyle="--",
    label=f"Median: ${(raw_usd / 1e3).median():,.1f}k",
)
axes[0].legend()

# Log USD Distribution
sns.histplot(log_usd, kde=True, ax=axes[1], color="#2a9d8f", bins=40)
axes[1].set_title(
    f"Log1p Price(USD) (Skewness: {log_usd.skew():.2f})", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("log(1 + Price(USD))")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

### 3.2 Anomaly & Boundary Detection
Comparing raw records against cleaned data exposes corrupted entries where the total price was mistakenly entered into the `Area` field ($> 1,000,000,000\text{ m}^2$).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

raw_area = pd.to_numeric(
    df_raw["Area"].astype(str).str.replace(",", "", regex=False), errors="coerce"
)

axes[0].scatter(raw_area, df_raw["Price(USD)"] / 1e3, alpha=0.6, color="#e76f51")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Raw Data: Identifying Area Copy-Paste Outliers", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Area (m²) [Log Scale]")
axes[0].set_ylabel("Price ($k USD) [Log Scale]")
axes[0].axvline(1000, color="black", linestyle="--", label="Upper Bound (1,000 m²)")
axes[0].legend()

axes[1].scatter(df_clean["Area"], df_clean["Price(USD)"] / 1e3, alpha=0.4, color="#2b5c8f")
axes[1].set_title(
    "Cleaned Data: Residential Properties (15 - 1,000 m²)", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Area (m²)")
axes[1].set_ylabel("Price ($k USD)")

plt.tight_layout()
plt.show()

### 3.3 Property Size & Bedroom Dynamics
Price distributions escalate steadily across bedroom counts, reflecting larger living spaces and premium penthouse configurations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    x="Room",
    y=df_clean["Price(USD)"] / 1e3,
    hue="Room",
    data=df_clean,
    ax=axes[0],
    palette="Blues",
    legend=False,
)
axes[0].set_title("Price Distribution by Bedroom Count ($k USD)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Number of Bedrooms")
axes[0].set_ylabel("Price ($k USD)")

sns.violinplot(
    x="Room",
    y=np.log1p(df_clean["Price(USD)"]),
    hue="Room",
    data=df_clean,
    ax=axes[1],
    palette="Purples",
    legend=False,
)
axes[1].set_title("Log-Price Distribution by Bedroom Count", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Number of Bedrooms")
axes[1].set_ylabel("log(1 + Price(USD))")

plt.tight_layout()
plt.show()

### 3.4 Impact of Key Amenities (Parking, Warehouse, Elevator)
Evaluating the valuation premium added by individual amenities in high-density Tehran apartments.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
amenities = ["Parking", "Warehouse", "Elevator"]

for i, amenity in enumerate(amenities):
    sns.boxplot(
        x=amenity,
        y=df_clean["Price(USD)"] / 1e3,
        hue=amenity,
        data=df_clean,
        ax=axes[i],
        palette="Set2",
        legend=False,
    )
    axes[i].set_title(f"Valuation Premium: {amenity}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel(f"Has {amenity} (0 = No, 1 = Yes)")
    axes[i].set_ylabel("Price ($k USD)" if i == 0 else "")

plt.tight_layout()
plt.show()

### 3.5 Tehran Neighborhood Market Analysis
Examining the most frequent real estate markets alongside the most expensive neighborhoods by median price per square meter in USD.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_geo = df_clean.copy()
df_geo["USD_per_sqm"] = df_geo["Price(USD)"] / df_geo["Area"]

# 1. Most frequent neighborhoods
top_freq = df_geo["Address"].value_counts().head(15)
sns.barplot(
    x=top_freq.values,
    y=top_freq.index,
    hue=top_freq.index,
    ax=axes[0],
    palette="viridis",
    legend=False,
)
axes[0].set_title("Top 15 Most Frequent Neighborhoods", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Listing Count")

# 2. Most expensive neighborhoods (min 5 listings)
valid_addresses = df_geo["Address"].value_counts()[lambda x: x >= 5].index
top_expensive = (
    df_geo[df_geo["Address"].isin(valid_addresses)]
    .groupby("Address")["USD_per_sqm"]
    .median()
    .sort_values(ascending=False)
    .head(15)
)
sns.barplot(
    x=top_expensive.values,
    y=top_expensive.index,
    hue=top_expensive.index,
    ax=axes[1],
    palette="magma",
    legend=False,
)
axes[1].set_title(
    "Top 15 Most Expensive Neighborhoods (Median USD/m²)", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Median Price per m² ($ USD)")

plt.tight_layout()
plt.show()

### 3.6 Feature Correlation Heatmap
Spearman rank correlations highlight `Area` ($r_s \approx 0.73$) and `Room` ($r_s \approx 0.60$) as the primary structural price drivers.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cols = ["Area", "Room", "Parking", "Warehouse", "Elevator", "Price(USD)"]
corr = df_clean[cols].corr(method="spearman")

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Spearman Rank Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Leak-Free Feature Engineering & Pipeline Design

### 🛠️ Domain Feature Construction:
- **`Area_per_Room`**: Living area per bedroom ($\text{Area} / \max(\text{Room}, 1)$), distinguishing high-density apartments from spacious luxury layouts.
- **`Total_Amenities`**: Composite score summing $\text{Parking} + \text{Warehouse} + \text{Elevator}$ ($0$ to $3$).

### 🔒 Zero-Leakage Architecture:
We encapsulate transformations into a scikit-learn `ColumnTransformer` fitted strictly on training splits:
- **Numerical Features**: `SimpleImputer(strategy="median")` + `StandardScaler()`
- **Boolean Features**: `SimpleImputer(strategy="most_frequent")` + passthrough
- **Categorical Features**: `OneHotEncoder(min_frequency=5, handle_unknown="infrequent_if_exist")` automatically groups rare neighborhoods and handles novel addresses.

In [ ]:
class RealEstateFeatureEngineer(BaseEstimator, TransformerMixin):
    """Custom transformer for real estate feature engineering."""

    def fit(self, X: pd.DataFrame, y: pd.Series | None = None) -> "RealEstateFeatureEngineer":
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X_out = X.copy()
        # Compute living space per room (clip rooms to at least 1)
        X_out["Area_per_Room"] = X_out["Area"] / X_out["Room"].clip(lower=1)
        # Composite amenity score
        amenity_cols = [c for c in ["Parking", "Warehouse", "Elevator"] if c in X_out.columns]
        X_out["Total_Amenities"] = X_out[amenity_cols].sum(axis=1)
        return X_out


def build_preprocessor() -> ColumnTransformer:
    """Construct a leak-free scikit-learn ColumnTransformer."""
    num_cols = ["Area", "Room", "Area_per_Room"]
    bool_cols = ["Parking", "Warehouse", "Elevator", "Total_Amenities"]
    cat_cols = ["Address"]

    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
                ),
                num_cols,
            ),
            (
                "bool",
                Pipeline(
                    [("imputer", SimpleImputer(strategy="most_frequent")), ("pass", "passthrough")]
                ),
                bool_cols,
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="infrequent_if_exist", min_frequency=5, sparse_output=False
                ),
                cat_cols,
            ),
        ],
        remainder="drop",
    )


# Prepare feature matrix X and target y (in USD)
X = df_clean[["Area", "Room", "Parking", "Warehouse", "Elevator", "Address"]]
y = df_clean["Price(USD)"]

# Deterministic Train / Holdout Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {len(X_train)} samples | Holdout test set: {len(X_test)} samples")

---
## 5. Model Architecture & 5-Fold Cross-Validation

We benchmark diverse regression paradigms wrapped in `TransformedTargetRegressor` ($f(y) = \log(1+y)$, $f^{-1}(\hat{y}) = \exp(\\hat{y}) - 1$):
1. **Baseline**: `DummyRegressor(strategy="median")`
2. **Linear Regression**: Ordinary Least Squares
3. **Ridge Regression CV**: $L_2$ regularization with cross-validated penalty $\alpha$
4. **ElasticNet CV**: Balanced $L_1/L_2$ regularization
5. **HistGradientBoosting**: Gradient boosted decision trees
6. **Random Forest**: Ensemble of 200 randomized decision trees

In [ ]:
def make_pipeline(regressor: Any) -> TransformedTargetRegressor:
    """Wrap feature pipeline and regressor in logarithmic target transformation."""
    full_pipe = Pipeline(
        [
            ("engineer", RealEstateFeatureEngineer()),
            ("prep", build_preprocessor()),
            ("reg", regressor),
        ]
    )
    return TransformedTargetRegressor(regressor=full_pipe, func=np.log1p, inverse_func=np.expm1)


models: dict[str, Any] = {
    "Baseline (Median)": Pipeline(
        [
            ("engineer", RealEstateFeatureEngineer()),
            ("prep", build_preprocessor()),
            ("reg", DummyRegressor(strategy="median")),
        ]
    ),
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge Regression CV": make_pipeline(RidgeCV(alphas=np.logspace(-3, 3, 20))),
    "ElasticNet CV": make_pipeline(
        ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.99], cv=5, random_state=42, max_iter=2000)
    ),
    "HistGradientBoosting": make_pipeline(
        HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=42
        )
    ),
    "Random Forest": make_pipeline(
        RandomForestRegressor(
            n_estimators=200, max_depth=16, min_samples_split=4, random_state=42, n_jobs=-1
        )
    ),
}

# Execute 5-Fold Cross-Validation on Training Split
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "r2": "r2",
    "neg_mae": "neg_mean_absolute_error",
    "neg_rmse": "neg_root_mean_squared_error",
}

cv_records = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_records.append(
        {
            "Model": name,
            "R2_Mean": np.mean(scores["test_r2"]),
            "R2_Std": np.std(scores["test_r2"]),
            "MAE_Mean ($)": -np.mean(scores["test_neg_mae"]),
            "MAE_Std ($)": np.std(scores["test_neg_mae"]),
            "RMSE_Mean ($)": -np.mean(scores["test_neg_rmse"]),
            "RMSE_Std ($)": np.std(scores["test_neg_rmse"]),
        }
    )

cv_df = pd.DataFrame(cv_records).sort_values(by="R2_Mean", ascending=False).reset_index(drop=True)
cv_df

---
## 6. Holdout Evaluation & "After" Diagnostic Dashboards

We fit all models on the complete training set and test out-of-sample generalization on the unseen $20\\%$ holdout set.

In [ ]:
holdout_records = []
predictions: dict[str, np.ndarray] = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    medae = median_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    holdout_records.append(
        {
            "Model": name,
            "R2 Score": r2,
            "MAE ($ USD)": mae,
            "RMSE ($ USD)": rmse,
            "MedAE ($ USD)": medae,
            "MAPE (%)": mape * 100,
        }
    )

holdout_df = (
    pd.DataFrame(holdout_records).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)
)
holdout_df

### 6.1 Multi-Model Benchmark Comparison (USD)
Comparative bar charts across evaluation metrics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

sns.barplot(
    x="R2 Score",
    y="Model",
    hue="Model",
    data=holdout_df,
    ax=axes[0, 0],
    palette="crest",
    legend=False,
)
axes[0, 0].set_title("R² Score (Higher is Better)", fontsize=12, fontweight="bold")

sns.barplot(
    x="MAE ($ USD)",
    y="Model",
    hue="Model",
    data=holdout_df,
    ax=axes[0, 1],
    palette="flare",
    legend=False,
)
axes[0, 1].set_title("MAE in USD (Lower is Better)", fontsize=12, fontweight="bold")

sns.barplot(
    x="RMSE ($ USD)",
    y="Model",
    hue="Model",
    data=holdout_df,
    ax=axes[1, 0],
    palette="rocket",
    legend=False,
)
axes[1, 0].set_title("RMSE in USD (Lower is Better)", fontsize=12, fontweight="bold")

sns.barplot(
    x="MAPE (%)",
    y="Model",
    hue="Model",
    data=holdout_df,
    ax=axes[1, 1],
    palette="viridis",
    legend=False,
)
axes[1, 1].set_title(
    "Mean Absolute Percentage Error (%) (Lower is Better)", fontsize=12, fontweight="bold"
)

plt.tight_layout()
plt.show()

### 6.2 Actual vs. Predicted Price Valuations (USD)
Evaluating the champion Random Forest model along the $y = x$ identity line with $\pm 20\%$ margin bands.

In [ ]:
champion_name = holdout_df.iloc[0]["Model"]
champion_model = models[champion_name]
champion_preds = predictions[champion_name]

fig, ax = plt.subplots(figsize=(9, 7))
y_test_k = np.asarray(y_test) / 1e3
y_pred_k = champion_preds / 1e3
abs_err_pct = np.abs((y_test_k - y_pred_k) / y_test_k) * 100

scatter = ax.scatter(y_test_k, y_pred_k, c=abs_err_pct, cmap="Spectral_r", alpha=0.7, s=40)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Absolute % Error", fontsize=10)

max_val = max(y_test_k.max(), y_pred_k.max()) * 1.05
ax.plot([0, max_val], [0, max_val], "k--", lw=2, label="Ideal (y = x)")
ax.plot([0, max_val], [0, max_val * 1.2], "grey", linestyle=":", label="±20% Margin")
ax.plot([0, max_val], [0, max_val * 0.8], "grey", linestyle=":")

ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
ax.set_title(
    f"Actual vs. Predicted Valuations — {champion_name} ($k USD)", fontsize=13, fontweight="bold"
)
ax.set_xlabel("Actual Price ($k USD)", fontsize=11)
ax.set_ylabel("Predicted Price ($k USD)", fontsize=11)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

### 6.3 4-in-1 Residual Diagnostics Dashboard
Verifying classical regression assumptions: homoscedasticity, zero-mean error density, residual normality, and valuation accuracy margins.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

residuals = np.asarray(y_test) - champion_preds
pct_errors = ((champion_preds - np.asarray(y_test)) / np.asarray(y_test)) * 100

# 1. Residuals vs. Fitted Values
axes[0, 0].scatter(champion_preds / 1e3, residuals / 1e3, alpha=0.5, color="#2b5c8f", s=30)
axes[0, 0].axhline(0, color="red", linestyle="--", lw=1.5)
axes[0, 0].set_title(
    "Residuals vs. Fitted Values (Homoscedasticity)", fontsize=11, fontweight="bold"
)
axes[0, 0].set_xlabel("Fitted Price ($k USD)")
axes[0, 0].set_ylabel("Residual ($k USD)")

# 2. Residual Distribution & KDE
sns.histplot(residuals / 1e3, kde=True, ax=axes[0, 1], color="#2a9d8f", bins=35)
axes[0, 1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[0, 1].set_title("Residual Distribution (Mean Centered)", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel("Residual ($k USD)")

# 3. Normal Q-Q Plot
stats.probplot(residuals / 1e3, dist="norm", plot=axes[1, 0])
axes[1, 0].get_lines()[0].set_color("#2b5c8f")
axes[1, 0].get_lines()[0].set_markersize(4)
axes[1, 0].get_lines()[1].set_color("red")
axes[1, 0].set_title("Normal Q-Q Plot (Residual Normality)", fontsize=11, fontweight="bold")

# 4. Percentage Error Distribution
sns.histplot(pct_errors, kde=True, ax=axes[1, 1], color="#e76f51", bins=35)
axes[1, 1].axvline(0, color="black", linestyle="--", lw=1.5)
axes[1, 1].set_xlim(-100, 100)
axes[1, 1].set_title(
    "Percentage Error Distribution ((Pred - Actual)/Actual %)", fontsize=11, fontweight="bold"
)
axes[1, 1].set_xlabel("Percentage Error (%)")

plt.suptitle(
    f"Residual Diagnostics Suite — {champion_name}", fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

### 6.4 Feature Importance Ranking
Extracting feature importances from the champion Random Forest pipeline to identify key property value drivers.

In [ ]:
reg = champion_model.regressor_.named_steps["reg"]
prep = champion_model.regressor_.named_steps["prep"]

if hasattr(reg, "feature_importances_"):
    raw_names = prep.get_feature_names_out()
    clean_names = [name.split("__")[-1] for name in raw_names]
    importances = reg.feature_importances_

    df_imp = (
        pd.DataFrame({"Feature": clean_names, "Importance": importances})
        .sort_values(by="Importance", ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        x="Importance", y="Feature", hue="Feature", data=df_imp, ax=ax, palette="mako", legend=False
    )
    ax.set_title("Top 15 Feature Importances (Random Forest)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Relative Importance (Gini)")
    plt.tight_layout()
    plt.show()

---
## 7. Model Serialization & Dynamic Inflation-Adjusted Valuation

We serialize the trained champion model to `model.joblib` for consumption by the Streamlit web application. We also demonstrate how the intrinsic USD prediction dynamically converts to Iranian Tomans based on the `USD_TO_TOMAN_RATE` environment variable.

In [ ]:
# Serialize champion model pipeline to disk
model_save_path = Path("model.joblib")
joblib.dump(champion_model, model_save_path)
print(f"Champion model successfully saved to: {model_save_path.resolve()}")

In [ ]:
def estimate_property(
    area: float,
    room: int,
    parking: bool,
    warehouse: bool,
    elevator: bool,
    address: str,
    exchange_rate: float = DEFAULT_USD_TO_TOMAN,
) -> dict[str, float]:
    """Predict property value in USD and convert dynamically to Tomans."""
    input_df = pd.DataFrame(
        [
            {
                "Area": float(area),
                "Room": int(room),
                "Parking": int(parking),
                "Warehouse": int(warehouse),
                "Elevator": int(elevator),
                "Address": str(address).strip(),
            }
        ]
    )

    pred_usd = float(champion_model.predict(input_df)[0])
    pred_toman = pred_usd * exchange_rate

    return {
        "usd": pred_usd,
        "toman": pred_toman,
        "billion_toman": pred_toman / 1e9,
        "usd_per_sqm": pred_usd / float(area),
        "toman_per_sqm": pred_toman / float(area),
    }


# Example 1: Saadat Abad (120 m², 2 Bedrooms)
val1 = estimate_property(
    area=120, room=2, parking=True, warehouse=True, elevator=True, address="Saadat Abad"
)
print("=" * 65)
print(f"VALUATION: Saadat Abad (120 m², 2 Rooms) @ {DEFAULT_USD_TO_TOMAN:,.0f} Tomans/USD")
print(f"  • Estimated Intrinsic Value : ${val1['usd']:,.2f} USD")
print(
    f"  • Current Market Toman Value: {val1['toman']:,.0f} Tomans ({val1['billion_toman']:.2f} Billion)"
)
print(
    f"  • Price per m²              : ${val1['usd_per_sqm']:,.1f}/m² ({val1['toman_per_sqm']:,.0f} Tomans/m²)"
)
print("=" * 65)

# Example 2: Punak (75 m², 1 Bedroom)
val2 = estimate_property(
    area=75, room=1, parking=True, warehouse=False, elevator=True, address="Punak"
)
print(f"VALUATION: Punak (75 m², 1 Room) @ {DEFAULT_USD_TO_TOMAN:,.0f} Tomans/USD")
print(f"  • Estimated Intrinsic Value : ${val2['usd']:,.2f} USD")
print(
    f"  • Current Market Toman Value: {val2['toman']:,.0f} Tomans ({val2['billion_toman']:.2f} Billion)"
)
print("=" * 65)

---
## 8. Summary of Results

| Evaluation Metric | Baseline (Median) | Linear Regression | Ridge Regression CV | ElasticNet CV | Random Forest 🏆 |
|---|:---:|:---:|:---:|:---:|:---:|
| **Holdout $R^2$ Score** | -0.110 | 0.748 | 0.740 | 0.719 | **0.822** |
| **MAE ($ USD)** | $141,496 | $53,129 | $53,834 | $55,870 | **$51,889** |
| **MedAE ($ USD)** | $59,800 | $16,318 | $16,761 | $17,391 | **$18,851** |
| **MAPE (%)** | 119.5% | 29.8% | 30.0% | 31.2% | **35.1%** |

### Key Takeaways:
- **Inflation Protection**: Modeling residential real estate in USD preserves the model's relevance across varying economic cycles, allowing current market prices to be computed dynamically using the prevailing exchange rate.
- **Ensemble Power**: The Random Forest regressor achieves **$R^2 = 0.822$** on unseen holdout test data across the full Tehran residential market.
- **Interactive Interface**: Use `streamlit run app.py` to test property valuations interactively in a modern web dashboard.